# Ejercicio 3
Efecto de reescalar `x2`.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
from lista4.data import load_csv
from lista4.models import fit_linear, linear_predictions
from lista4.metrics import empirical_risk, absolute_prediction_difference

df = load_csv("experimento_3.csv")
df["z2"] = df["x2"] / 100

m_original = fit_linear(df, ["x1", "x2"])
m_escalado = fit_linear(df, ["x1", "z2"])

pred_original = linear_predictions(m_original, df, ["x1", "x2"])
pred_escalado = linear_predictions(m_escalado, df, ["x1", "z2"])

r_original = empirical_risk(df["y"], pred_original)
r_escalado = empirical_risk(df["y"], pred_escalado)

print("Modelo original")
print("beta0 =", m_original.intercept_)
print("beta1 =", m_original.coef_[0])
print("beta2 =", m_original.coef_[1])
print("Riesgo =", r_original)

print("\nModelo con z2 = x2/100")
print("gamma0 =", m_escalado.intercept_)
print("gamma1 =", m_escalado.coef_[0])
print("gamma2 =", m_escalado.coef_[1])
print("Riesgo =", r_escalado)


Modelo original
beta0 = -0.1014700964075996
beta1 = 5.012065732244869
beta2 = 0.050476925444612616
Riesgo = 0.9232252621022037

Modelo con z2 = x2/100
gamma0 = -0.10147009640759957
gamma1 = 5.01206573224487
gamma2 = 5.04769254446126
Riesgo = 0.9232252621022038



Ajustamos los dos modelos y los coeficientes quedan casi iguales, salvo el de
`x2`. El intercepto y el coeficiente de `x1` son idénticos en los dos modelos. Lo
único que cambia es el coeficiente de `x2`, que en el modelo con `z2` queda 100
veces más grande.

El riesgo empírico es exactamente el mismo en los dos modelos.

In [2]:
d = absolute_prediction_difference(pred_original, pred_escalado)

print("max_i d_i =", d.max())

comparaciones = pd.DataFrame({
    "comparacion": ["gamma0 vs beta0", "gamma1 vs beta1", "gamma2 vs 100*beta2"],
    "valor_izquierdo": [
        m_escalado.intercept_,
        m_escalado.coef_[0],
        m_escalado.coef_[1]
    ],
    "valor_derecho": [
        m_original.intercept_,
        m_original.coef_[0],
        100 * m_original.coef_[1]
    ]
})

comparaciones["error_absoluto"] = (
    comparaciones["valor_izquierdo"] -
    comparaciones["valor_derecho"]
).abs()

comparaciones


max_i d_i = 7.16093850883226e-15


,comparacion,valor_izquierdo,valor_derecho,error_absoluto
0,gamma0 vs beta0,-0.101470,-0.101470,2.775558e-17
1,gamma1 vs beta1,5.012066,5.012066,8.881784e-16
2,gamma2 vs 100*beta2,5.047693,5.047693,1.776357e-15


Calculamos la diferencia entre las predicciones de los dos modelos y el máximo
nos dio prácticamente cero. O sea, los dos modelos predicen exactamente lo mismo
en todas las observaciones. Eso nos confirma que reescalar `x2` no cambia nada
del modelo.

Comparamos los coeficientes y los errores absolutos nos dieron prácticamente
cero. Eso nos muestra que `gamma0` es igual a `beta0`, `gamma1` es igual a
`beta1`, y `gamma2` es igual a 100 por `beta2`. El único coeficiente que cambia
es el de `x2`, y cambia justo por el factor 100.

El valor absoluto de `gamma2` es como 100 veces el de `beta2`, pero las
predicciones de los dos modelos son iguales. Entonces nos damos cuenta de que el
tamaño de un coeficiente solo no dice nada. Si una variable está medida en otras
unidades, su coeficiente cambia para compensar y las predicciones quedan iguales.

In [3]:
tabla = pd.DataFrame({
    "coeficiente_beta": [
        m_original.intercept_,
        m_original.coef_[0],
        m_original.coef_[1]
    ],
    "coeficiente_gamma": [
        m_escalado.intercept_,
        m_escalado.coef_[0],
        m_escalado.coef_[1]
    ]
}, index=["intercepto", "x1", "x2/z2"])

tabla["riesgo_beta"] = r_original
tabla["riesgo_gamma"] = r_escalado
tabla["max_d_i"] = d.max()

tabla


,coeficiente_beta,coeficiente_gamma,riesgo_beta,riesgo_gamma,max_d_i
intercepto,-0.101470,-0.101470,0.923225,0.923225,7.160939e-15
x1,5.012066,5.012066,0.923225,0.923225,7.160939e-15
x2/z2,0.050477,5.047693,0.923225,0.923225,7.160939e-15


La tabla reúne los coeficientes, los riesgos y el máximo de `d_i` de los dos
modelos. Los riesgos son iguales y la diferencia entre predicciones es cero,
aunque los coeficientes de `x2` y `z2` sean muy distintos. Es lo mismo de antes:
el modelo no cambia, solo cambia cómo se escribe el coeficiente según la escala
de la variable.